In [11]:
from kafka import KafkaConsumer
import json
import pandas as pd
from sklearn.ensemble import IsolationForest
import warnings
import pickle

warnings.filterwarnings('ignore')

TOPIC_NAME = 'gcp_billing_events'
BATCH_SIZE = 100
MAX_HISTORY = 2000  

EXPECTED_CATEGORIES = [
    'service_BigQuery', 'service_Cloud Functions', 'service_Cloud Storage', 
    'service_Compute Engine', 'service_Kubernetes Engine'
]

In [12]:
def prepare_features(events_list):
    """
    Przyjmuje listę surowych zdarzeń i przygotowuje gotową tabelę (DataFrame) do treningu/predykcji.
    """
    df = pd.DataFrame(events_list)
    
 
    df_services = pd.get_dummies(df['service'], prefix='service')
    

    for col in EXPECTED_CATEGORIES:
        if col not in df_services.columns:
            df_services[col] = 0
            
    
    df_services = df_services[EXPECTED_CATEGORIES]
    
   
    X = pd.concat([df[['usage_amount', 'cost_usd', 'is_weekend']], df_services], axis=1).fillna(0)
    
    return X, df

In [13]:
print(f"🎧 Podłączam Konsumenta Isolation Forest do topicu '{TOPIC_NAME}'...")

consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers='broker:9092',
    group_id='finops-iforest-group',
    auto_offset_reset='latest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

model = IsolationForest(contamination=0.05, random_state=42)

new_events_buffer = []  
historical_window = [] 
batch_counter = 0

print("✅ Gotowe. Możesz przejść do uruchomienia pętli nasłuchującej.")

🎧 Podłączam Konsumenta Isolation Forest do topicu 'gcp_billing_events'...
✅ Gotowe. Możesz przejść do uruchomienia pętli nasłuchującej.


In [16]:
print("⏳ Oczekuję na dane... Buforowanie w toku.\n")

try:
    for message in consumer:
        event = message.value
        new_events_buffer.append(event)
        historical_window.append(event)
        
    
        if len(historical_window) > MAX_HISTORY:
            historical_window = historical_window[-MAX_HISTORY:]
            
       
        if len(new_events_buffer) >= BATCH_SIZE:
            batch_counter += 1
            
          
            X_history, _ = prepare_features(historical_window)
            X_new, df_new = prepare_features(new_events_buffer)
            
       
            model.fit(X_history)
            

            predictions = model.predict(X_new)
            
          
            anomalies_detected = sum([1 for p in predictions if p == -1])
            
           
            anomaly_scores = model.score_samples(X_new)
            
            print(f"📊 [Aktualizacja {batch_counter}] Model wytrenowany na bazie {len(X_history)} zdarzeń historycznych.")
            print(f"   🚨 Wykryto anomalii w nowej paczce: {anomalies_detected} / {BATCH_SIZE}")
            
           
            with open('finops_iforest_model.pkl', 'wb') as f:
                pickle.dump(model, f)
            print("   💾 Zapisano 'finops_iforest_model.pkl'.\n")
            
           
            new_events_buffer = []
            
except KeyboardInterrupt:
    print("\n⏹️ Nasłuch przerwany przez użytkownika.")
finally:
    consumer.close()

⏳ Oczekuję na dane... Buforowanie w toku.


⏹️ Nasłuch przerwany przez użytkownika.
